# Demo: Concatenation-Fusion Skin Lesion Pipeline

This notebook demonstrates the complete workflow using **concatenation fusion** for:

1. **MobileViT + BioClinicalBERT concat**
2. **ResNet50 + BioClinicalBERT concat**

For each backbone, the notebook shows how to run:

- Hyperparameter tuning with Optuna/TPE
- PAD-UFES closed-set training using all three text combinations:
  - `text_full`
  - `text_core`
  - `text_missing_explicit`
- Direct closed-set generalization to target datasets
- DANN domain adaptation
- Energy-based open-world evaluation
- Result collection and printed summary tables

Long training commands are controlled by `RUN_COMMANDS`. Keep it `False` to preview commands; set it to `True` to run.

## 1. Imports and notebook controls

In [2]:
from __future__ import annotations

import json
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Iterable

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Set True when ready to run tuning/training/evaluation.
RUN_COMMANDS = True

# Print every command before running.
PRINT_COMMANDS = True

# Stop immediately if a subprocess fails.
CHECK_SUBPROCESS = True

# N_TRIALS = 10          
# TRIAL_EPOCHS = 3      
# CLOSED_SET_EPOCHS = 50
# DANN_EPOCHS = 20

N_TRIALS = 2
TRIAL_EPOCHS = 1
CLOSED_SET_EPOCHS = 1
DANN_EPOCHS = 1

## 2. Project/package setup

Your scripts use package-relative imports such as `from .datasets import ...`, so the safest way to run them is as modules:

```bash
python -m your_package.train_closed_set
```

Update `PACKAGE_NAME` and `PROJECT_ROOT`.

In [3]:
PROJECT_ROOT = Path(r"D:/Deep Learning/skin_lesion_openworld/skin_lesion_openworld")

# Your .py files are directly inside:
# D:/Deep Learning/skin_lesion_openworld/skin_lesion_openworld/src
PACKAGE_NAME = "src"

PACKAGE_ROOT = PROJECT_ROOT / PACKAGE_NAME

# Keep this True because your scripts use relative imports:
# from .datasets import ...
RUN_AS_MODULE = True

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("RUN_AS_MODULE:", RUN_AS_MODULE)

PROJECT_ROOT: D:\Deep Learning\skin_lesion_openworld\skin_lesion_openworld
PACKAGE_ROOT: D:\Deep Learning\skin_lesion_openworld\skin_lesion_openworld\src
RUN_AS_MODULE: True


## 3. Required files and concatenation-fusion validation

This checks that the concat model families are exposed:

- `mobilevit_concat`
- `resnet50_concat`

In [4]:
REQUIRED_FILES = [
    "config.py",
    "preprocessing.py",
    "datasets.py",
    "models.py",
    "metrics.py",
    "transforms.py",
    "visualization.py",
    "utils.py",
    "train_closed_set.py",
    "evaluate_closed_set_transfer.py",
    "train_dann.py",
    "evaluate_openworld.py",
    "tune_hyperparameters.py",
]

def package_file(name: str) -> Path:
    return PACKAGE_ROOT / name if RUN_AS_MODULE else PROJECT_ROOT / name

def validate_project_files() -> None:
    missing = [name for name in REQUIRED_FILES if not package_file(name).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required files: "
            + ", ".join(missing)
            + f"\nChecked: {PACKAGE_ROOT if RUN_AS_MODULE else PROJECT_ROOT}"
        )

    models_text = package_file("models.py").read_text(encoding="utf-8")
    required_model_markers = [
        "ConcatenationFusionHead",
        "mobilevit_concat",
        "resnet50_concat",
    ]
    missing_model_markers = [
        marker for marker in required_model_markers if marker not in models_text
    ]
    if missing_model_markers:
        raise ValueError(
            "models.py is not concat-ready. Missing: "
            + ", ".join(missing_model_markers)
            + "\nReplace models.py with the concat-ready models_corrected.py."
        )

    for script_name in [
        "train_closed_set.py",
        "train_dann.py",
        "evaluate_openworld.py",
        "evaluate_closed_set_transfer.py",
        "tune_hyperparameters.py",
    ]:
        text = package_file(script_name).read_text(encoding="utf-8")
        for model_key in ["mobilevit_concat", "resnet50_concat"]:
            if model_key not in text:
                raise ValueError(
                    f"{script_name} does not expose {model_key}. "
                    "Use the corrected concat-ready version."
                )

    print("Project files found and concatenation fusion is exposed.")

# Run this after setting PROJECT_ROOT and PACKAGE_NAME.
validate_project_files()

Project files found and concatenation fusion is exposed.


## 4. Dataset paths and output locations

In [30]:
# ----------------------------
# Raw PAD-UFES paths
# ----------------------------
PADUFES_CSV = Path(r"D:/Deep Learning/metadata.csv")
PADUFES_IMAGE_DIR = Path(r"D:/Deep Learning/images")

# ----------------------------
# Standardized split CSV
# ----------------------------
OUTPUT_ROOT = Path(r"D:/Deep Learning/output_concat_demo")
STANDARDIZED_DIR = OUTPUT_ROOT / "standardized"
STANDARDIZED_CSV = STANDARDIZED_DIR / "all_preprocessed_splits_standardized_text.csv"

# ----------------------------
# Target image roots
# ----------------------------
SOURCE_DATASET = "PAD-UFES-20"

TARGETS = {
    "ISIC 2019": [
        Path(r"D:/Deep Learning/ISIC_2019_Training_Input"),
        Path(r"D:/Deep Learning/ISIC_2019_Test_Input"),
    ],
    "MCR-SL": [
        Path(r"D:/Deep Learning/MCR-SL_dataset/dermoscopic"),
        Path(r"D:/Deep Learning/MCR-SL_dataset/images"),
        Path(r"D:/Deep Learning/MCR-SL_dataset"),
    ],
}

# ----------------------------
# Optional raw metadata paths for preprocessing
# ----------------------------
ISIC_TRAIN_GT = Path(r"D:/Deep Learning/ISIC_2019_Training_GroundTruth.csv")
ISIC_TRAIN_META = Path(r"D:/Deep Learning/ISIC_2019_Training_Metadata.csv")
ISIC_TEST_GT = Path(r"D:/Deep Learning/ISIC_2019_Test_GroundTruth.csv")
ISIC_TEST_META = Path(r"D:/Deep Learning/ISIC_2019_Test_Metadata.csv")

MCR_UNIFIED_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/unified_diagnosis.xlsx")
MCR_LESION_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/lesion.xlsx")
MCR_SUBJECT_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/subject.xlsx")
MCR_IMAGE_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/image.xlsx")
MCR_DERM_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/dermatology_diagnosis.xlsx")
MCR_HISTO_XLSX = Path(r"D:/Deep Learning/MCR-SL_dataset/histopathology_diagnosis.xlsx")

# ----------------------------
# Output folders
# ----------------------------
TUNING_OUT = OUTPUT_ROOT / "01_tuning"
CLOSED_SET_OUT = OUTPUT_ROOT / "02_closed_set"
DIRECT_TRANSFER_OUT = OUTPUT_ROOT / "03_direct_transfer"
DANN_OUT = OUTPUT_ROOT / "04_dann"
OPENWORLD_OUT = OUTPUT_ROOT / "05_open_world"

TEXT_COLS = ["text_full", "text_core", "text_missing_explicit"]

# Short demo settings. Increase for final runs.
N_TRIALS = 10
TRIAL_EPOCHS = 3
CLOSED_SET_EPOCHS = 30
DANN_EPOCHS = 15
BATCH_SIZE_FALLBACK = 16
ENERGY_TEMPERATURE = 1.0

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

Output root: D:\Deep Learning\output_concat_demo


## 5. Command helpers

In [31]:
def module_or_script_cmd(module_name: str) -> list[str]:
    if RUN_AS_MODULE:
        return [sys.executable, "-m", f"{PACKAGE_NAME}.{module_name}"]
    return [sys.executable, str(PROJECT_ROOT / f"{module_name}.py")]

import os
import subprocess

def run_command(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess | None:
    if PRINT_COMMANDS:
        print("\n" + "=" * 120)
        print(" ".join(map(str, cmd)))
        print("=" * 120)

    if not RUN_COMMANDS:
        print("RUN_COMMANDS=False, command preview only.")
        return None

    env = os.environ.copy()

    # Important:
    # PROJECT_ROOT lets Python run modules as src.train_closed_set.
    # PACKAGE_ROOT lets absolute imports like "from config import ..." work.

    extra_python_paths = [
        str(PACKAGE_ROOT),  # src folder first, so "from config import ..." finds src/config.py
        str(PROJECT_ROOT),
    ]

    old_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = os.pathsep.join(
        extra_python_paths + ([old_pythonpath] if old_pythonpath else [])
    )

    old_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = os.pathsep.join(
        extra_python_paths + ([old_pythonpath] if old_pythonpath else [])
    )

    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd or PROJECT_ROOT),
        check=False,
        text=True,
        capture_output=True,
        env=env,
    )

    print("\nSTDOUT:")
    print(result.stdout)

    print("\nSTDERR:")
    print(result.stderr)

    if CHECK_SUBPROCESS and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode,
            result.args,
            output=result.stdout,
            stderr=result.stderr,
        )

    return result

def read_json(path: Path) -> dict:
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def find_first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

## 6. Optional preprocessing

Run this only if you still need to create `all_preprocessed_splits_standardized_text.csv`.

If the CSV already exists, skip this section.

In [7]:
# Optional preprocessing.
# Uncomment and adjust imports/package name if needed.
#
# from skin_lesion_owda.preprocessing import prepare_all_standardized_splits
#
# STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)
#
# standardized_df = prepare_all_standardized_splits(
#     output_dir=STANDARDIZED_DIR,
#     padufes_csv=PADUFES_CSV,
#     isic_train_groundtruth_csv=ISIC_TRAIN_GT,
#     isic_train_metadata_csv=ISIC_TRAIN_META,
#     isic_test_groundtruth_csv=ISIC_TEST_GT,
#     isic_test_metadata_csv=ISIC_TEST_META,
#     mcr_unified_xlsx=MCR_UNIFIED_XLSX,
#     mcr_lesion_xlsx=MCR_LESION_XLSX,
#     mcr_subject_xlsx=MCR_SUBJECT_XLSX,
#     mcr_image_xlsx=MCR_IMAGE_XLSX,
#     mcr_dermatology_diagnosis_xlsx=MCR_DERM_XLSX,
#     mcr_histopathology_diagnosis_xlsx=MCR_HISTO_XLSX,
#     write_individual_csvs=True,
#     print_reports=True,
# )
#
# display(standardized_df.head())
# display(standardized_df.groupby(["dataset", "split", "known_unknown"]).size())

## 7. Hyperparameter tuning helpers

Each backbone is tuned separately. The tuner searches over the three text combinations.

In [32]:
def tune_closed_set(model_family: str) -> Path:
    out_dir = TUNING_OUT / model_family / "closed_set"

    cmd = [
        *module_or_script_cmd("tune_hyperparameters"),
        "--study", "closed_set",
        "--padufes-csv", str(PADUFES_CSV),
        "--padufes-image-dir", str(PADUFES_IMAGE_DIR),
        "--output-dir", str(out_dir),
        "--n-trials", str(N_TRIALS),
        "--trial-epochs", str(TRIAL_EPOCHS),
        "--model-families", model_family,
        "--text-cols", *TEXT_COLS,
        "--batch-sizes", "8", "16", "32",
        "--fusion-dims", "128", "256", "384", "512",
        "--use-sampler",
        "--use-weighted-loss",
    ]

    run_command(cmd)
    return out_dir

def load_best_hyperparameters(tuning_dir: Path) -> dict:
    path = tuning_dir / "best_hyperparameters.json"
    if not path.exists():
        print("Best hyperparameter file not found yet:", path)
        return {}

    best = read_json(path)
    print(json.dumps(best, indent=2))
    return best.get("best_params", {})

def print_tuning_trials(tuning_dir: Path):
    trials_path = tuning_dir / "all_trials.csv"
    trials = safe_read_csv(trials_path)
    if trials.empty:
        print("No tuning trials found yet:", trials_path)
        return

    display(trials.sort_values("value", ascending=False).head(10))

## 8. Apply selected tuning results to `config.py`

The training scripts read some values from `config.py`.

This helper updates common defaults before launching subprocess training. It creates a backup named `config.py.demo_backup`.

Applied values when present:

- `lr`
- `weight_decay`
- `fusion_dim`
- `num_heads`
- `balance_beta`

`batch_size` is passed directly through CLI.

In [33]:
def update_config_default(config_text: str, key: str, value) -> str:
    pattern = rf"(^\s*{re.escape(key)}\s*:\s*[^=]+=\s*)(.+?)(\s*(?:#.*)?$)"
    replacement_value = repr(value)

    updated, n = re.subn(
        pattern,
        rf"\g<1>{replacement_value}\g<3>",
        config_text,
        flags=re.MULTILINE,
    )

    if n == 0:
        print(f"Warning: did not find config key `{key}`.")
    return updated

def apply_best_params_to_config(best_params: dict):
    if not best_params:
        print("No best params supplied; using existing config.py.")
        return

    config_path = package_file("config.py")
    backup_path = config_path.with_suffix(".py.demo_backup")

    if not backup_path.exists():
        shutil.copy2(config_path, backup_path)
        print("Created config backup:", backup_path)

    text = config_path.read_text(encoding="utf-8")

    for key in ["lr", "weight_decay", "fusion_dim", "num_heads", "balance_beta"]:
        if key in best_params:
            text = update_config_default(text, key, best_params[key])

    config_path.write_text(text, encoding="utf-8")
    print("Updated config.py with selected tuning values.")

def batch_size_from_params(best_params: dict) -> int:
    return int(best_params.get("batch_size", BATCH_SIZE_FALLBACK))

## 9. Pipeline command builders

In [34]:
def closed_set_checkpoint(model_family: str, text_col: str) -> Path:
    candidates = [
        CLOSED_SET_OUT / model_family / text_col / "best.pt",
        CLOSED_SET_OUT / model_family / text_col / f"{model_family}_{text_col}_best.pt",
    ]

    found = find_first_existing(candidates)
    if found is not None:
        return found

    search_root = CLOSED_SET_OUT / model_family / text_col
    matches = list(search_root.rglob("*best*.pt")) if search_root.exists() else []
    if matches:
        return matches[0]

    return candidates[0]

def dann_checkpoint(model_family: str, text_col: str, target_dataset: str) -> Path:
    target_name = target_dataset.replace("/", "_").replace("\\", "_").replace(" ", "_").replace("-", "_")

    candidates = [
        DANN_OUT / model_family / text_col / target_name / "best_dann.pt",
        DANN_OUT / model_family / text_col / target_dataset / "best_dann.pt",
    ]

    found = find_first_existing(candidates)
    if found is not None:
        return found

    base = DANN_OUT / model_family / text_col
    matches = list(base.rglob("*best*dann*.pt")) if base.exists() else []
    if matches:
        return matches[0]

    return candidates[0]

def run_closed_set_training(model_family: str, batch_size: int):
    cmd = [
        *module_or_script_cmd("train_closed_set"),
        "--padufes-csv", str(PADUFES_CSV),
        "--padufes-image-dir", str(PADUFES_IMAGE_DIR),
        "--output-dir", str(CLOSED_SET_OUT),
        "--model-family", model_family,
        "--text-col", "all",
        "--batch-size", str(batch_size),
        "--epochs", str(CLOSED_SET_EPOCHS),
    ]
    run_command(cmd)

def run_direct_generalization(
    model_family: str,
    text_col: str,
    target_dataset: str,
    target_roots: list[Path],
    batch_size: int,
):
    ckpt = closed_set_checkpoint(model_family, text_col)

    cmd = [
        *module_or_script_cmd("evaluate_closed_set_transfer"),
        "--standardized-csv", str(STANDARDIZED_CSV),
        "--target-dataset", target_dataset,
        "--target-image-roots", *map(str, target_roots),
        "--checkpoint", str(ckpt),
        "--output-dir", str(DIRECT_TRANSFER_OUT),
        "--model-family", model_family,
        "--text-col", text_col,
        "--batch-size", str(batch_size),
    ]
    run_command(cmd)

def run_dann(
    model_family: str,
    text_col: str,
    target_dataset: str,
    target_roots: list[Path],
    batch_size: int,
):
    ckpt = closed_set_checkpoint(model_family, text_col)

    cmd = [
        *module_or_script_cmd("train_dann"),
        "--standardized-csv", str(STANDARDIZED_CSV),
        "--source-dataset", "PAD-UFES",
        "--target-dataset", target_dataset,
        "--source-image-roots", str(PADUFES_IMAGE_DIR),
        "--target-image-roots", *map(str, target_roots),
        "--checkpoint", str(ckpt),
        "--output-dir", str(DANN_OUT),
        "--model-family", model_family,
        "--text-col", text_col,
        "--batch-size", str(batch_size),
        "--epochs", str(DANN_EPOCHS),
        "--energy-temperature", str(ENERGY_TEMPERATURE),
    ]
    run_command(cmd)

def run_open_world(
    model_family: str,
    text_col: str,
    target_dataset: str,
    target_roots: list[Path],
    batch_size: int,
):
    ckpt = dann_checkpoint(model_family, text_col, target_dataset)

    cmd = [
        *module_or_script_cmd("evaluate_openworld"),
        "--standardized-csv", str(STANDARDIZED_CSV),
        "--target-dataset", target_dataset,
        "--target-image-roots", *map(str, target_roots),
        "--checkpoint", str(ckpt),
        "--output-dir", str(OPENWORLD_OUT),
        "--model-family", model_family,
        "--text-col", text_col,
        "--batch-size", str(batch_size),
        "--energy-temperature", str(ENERGY_TEMPERATURE),
        "--dann",
    ]
    run_command(cmd)

## 10. Result collection helpers

In [35]:
def collect_csv_files(root: Path, patterns: list[str]) -> pd.DataFrame:
    rows = []

    for pattern in patterns:
        for path in root.rglob(pattern) if root.exists() else []:
            try:
                df = pd.read_csv(path)
                if df.empty:
                    continue
                df["source_file"] = str(path)
                rows.append(df)
            except Exception as exc:
                print("Could not read", path, "because", exc)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True, sort=False)

def collect_json_metrics(root: Path, patterns: list[str]) -> pd.DataFrame:
    rows = []

    for pattern in patterns:
        for path in root.rglob(pattern) if root.exists() else []:
            try:
                data = read_json(path)
                flat = {"source_file": str(path)}

                def flatten(prefix, obj):
                    if isinstance(obj, dict):
                        for k, v in obj.items():
                            flatten(f"{prefix}{k}.", v)
                    elif isinstance(obj, (int, float, str, bool)) or obj is None:
                        flat[prefix[:-1]] = obj

                flatten("", data)
                rows.append(flat)
            except Exception as exc:
                print("Could not read", path, "because", exc)

    return pd.DataFrame(rows)

def choose_best_row(df: pd.DataFrame):
    metric_priority = [
        "test_macro_f1",
        "test_f1_macro",
        "macro_f1",
        "f1_macro",
        "val_macro_f1",
        "val_f1_macro",
        "accuracy",
        "test_accuracy",
    ]

    metric_col = None
    for col in metric_priority:
        if col in df.columns:
            metric_col = col
            break

    if metric_col is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        if not numeric_cols:
            raise ValueError("No numeric metric column found in summary CSV.")
        metric_col = numeric_cols[0]

    best_idx = df[metric_col].astype(float).idxmax()
    return df.loc[[best_idx]].copy(), metric_col


def print_closed_set_results(model_family: str):
    print(f"\nClosed-set best result for {model_family}")

    # Load only summary CSVs, not classification reports.
    summary_files = list((CLOSED_SET_OUT).glob("closed_set_summary.csv"))
    summary_files += list((CLOSED_SET_OUT).glob("*_closed_set_summary.csv"))

    # Also check model-specific nested folders, just in case.
    summary_files += list((CLOSED_SET_OUT / model_family).rglob("closed_set_summary.csv"))
    summary_files += list((CLOSED_SET_OUT / model_family).rglob("*_closed_set_summary.csv"))

    summary_files = sorted(set(summary_files))

    if not summary_files:
        print("No closed-set summary CSV found yet.")
        return pd.DataFrame()

    rows = []
    for path in summary_files:
        df = pd.read_csv(path)
        if not df.empty:
            df["source_file"] = str(path)
            rows.append(df)

    if not rows:
        print("Closed-set summary files were found but empty.")
        return pd.DataFrame()

    summary_df = pd.concat(rows, ignore_index=True, sort=False)

    best_df, metric_col = choose_best_row(summary_df)

    best_out = CLOSED_SET_OUT / model_family / "best_closed_set_result.csv"
    best_out.parent.mkdir(parents=True, exist_ok=True)
    best_df.to_csv(best_out, index=False)

    print(f"Selected best row using metric: {metric_col}")
    print(f"Saved best closed-set result to: {best_out}")

    display(best_df)

    return best_df

def print_direct_transfer_results(model_family: str):
    print(f"\nDirect generalization results for {model_family}")
    root = DIRECT_TRANSFER_OUT / model_family
    df = collect_csv_files(root, ["*summary*.csv", "*classification_report.csv"])
    if df.empty:
        print("No direct transfer summaries/reports found yet.")
    else:
        display(df.head(30))

def _select_best_metric_column(df: pd.DataFrame):
    metric_priority = [
        "test_macro_f1",
        "target_test_known.macro_f1",
        "target_test_known_macro_f1",
        "openworld_target_test.open_macro_f1",
        "open_macro_f1",
        "macro_f1",
        "f1_macro",
        "weighted_f1",
        "accuracy",
        "open_accuracy",
    ]

    for col in metric_priority:
        if col in df.columns:
            return col

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    return numeric_cols[0] if numeric_cols else None


def _display_best_summary(
    *,
    title: str,
    root: Path,
    output_name: str,
    patterns: list[str] | None = None,
):
    print(f"\n{title}")

    patterns = patterns or ["*summary*.csv"]
    df = collect_csv_files(root, patterns)

    if df.empty:
        print(f"No summary CSV found under: {root}")
        return pd.DataFrame()

    metric_col = _select_best_metric_column(df)

    if metric_col is None:
        print("Summary found, but no numeric metric column was available.")
        display(df.head(10))
        return df

    best_idx = df[metric_col].astype(float).idxmax()
    best_df = df.loc[[best_idx]].copy()

    out_path = root / output_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    best_df.to_csv(out_path, index=False)

    print(f"Selected best row using metric: {metric_col}")
    print(f"Saved: {out_path}")

    display(best_df)
    return best_df


def print_direct_transfer_results(model_family: str):
    root = DIRECT_TRANSFER_OUT / model_family

    return _display_best_summary(
        title=f"Best direct generalization result for {model_family}",
        root=root,
        output_name="best_direct_transfer_result.csv",
        patterns=["*summary*.csv"],
    )


def print_dann_results(model_family: str):
    root = DANN_OUT / model_family

    return _display_best_summary(
        title=f"Best DANN result for {model_family}",
        root=root,
        output_name="best_dann_result.csv",
        patterns=["*summary*.csv"],
    )


def print_open_world_results(model_family: str):
    print(f"\nBest open-world result for {model_family}")

    root = OPENWORLD_OUT / model_family

    # Prefer summary CSVs if present.
    csv_df = collect_csv_files(root, ["*summary*.csv"])

    if not csv_df.empty:
        metric_col = _select_best_metric_column(csv_df)

        if metric_col is not None:
            best_idx = csv_df[metric_col].astype(float).idxmax()
            best_df = csv_df.loc[[best_idx]].copy()

            out_path = root / "best_open_world_result.csv"
            out_path.parent.mkdir(parents=True, exist_ok=True)
            best_df.to_csv(out_path, index=False)

            print(f"Selected best CSV row using metric: {metric_col}")
            print(f"Saved: {out_path}")
            display(best_df)
            return best_df

    # Fallback to JSON metrics.
    json_df = collect_json_metrics(root, ["*metrics*.json", "openworld_metrics.json"])

    if json_df.empty:
        print("No open-world summary or metrics files found yet.")
        return pd.DataFrame()

    important_cols = [
        c for c in json_df.columns
        if any(key in c for key in [
            "open_macro_f1",
            "open_weighted_f1",
            "unknown_auroc",
            "unknown_recall",
            "unknown_f1",
            "oscr",
            "threshold",
            "scoring_method",
        ])
    ]

    metric_col = _select_best_metric_column(json_df[important_cols]) if important_cols else None

    if metric_col is None:
        display(json_df.head(10))
        return json_df

    best_idx = json_df[metric_col].astype(float).idxmax()
    best_df = json_df.loc[[best_idx]].copy()

    out_path = root / "best_open_world_result.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    best_df.to_csv(out_path, index=False)

    print(f"Selected best JSON row using metric: {metric_col}")
    print(f"Saved: {out_path}")

    cols_to_show = ["source_file", *important_cols]
    cols_to_show = [c for c in cols_to_show if c in best_df.columns]
    display(best_df[cols_to_show])

    return best_df

def print_all_results(model_family: str):
    print_closed_set_results(model_family)
    print_direct_transfer_results(model_family)
    print_dann_results(model_family)
    print_open_world_results(model_family)

## 11. Full pipeline function

In [37]:
def tune_closed_set(model_family: str, n_trials: int, trial_epochs: int) -> Path:
    out_dir = TUNING_OUT / model_family / "closed_set"

    cmd = [
        *module_or_script_cmd("tune_hyperparameters"),
        "--study", "closed_set",
        "--padufes-csv", str(PADUFES_CSV),
        "--padufes-image-dir", str(PADUFES_IMAGE_DIR),
        "--output-dir", str(out_dir),
        "--n-trials", str(n_trials),
        "--trial-epochs", str(trial_epochs),
        "--model-families", model_family,
        "--text-cols", *TEXT_COLS,
        "--batch-sizes", "8", "16", "32",
        "--fusion-dims", "128", "256", "384", "512",
        "--use-sampler",
        "--use-weighted-loss",
    ]

    run_command(cmd)
    return out_dir


def run_closed_set_training(model_family: str, batch_size: int, epochs: int):
    cmd = [
        *module_or_script_cmd("train_closed_set"),
        "--padufes-csv", str(PADUFES_CSV),
        "--padufes-image-dir", str(PADUFES_IMAGE_DIR),
        "--output-dir", str(CLOSED_SET_OUT),
        "--model-family", model_family,
        "--text-col", "all",
        "--batch-size", str(batch_size),
        "--epochs", str(epochs),
    ]
    run_command(cmd)


def run_dann(
    model_family: str,
    text_col: str,
    target_dataset: str,
    target_roots: list[Path],
    batch_size: int,
    epochs: int,
):
    ckpt = closed_set_checkpoint(model_family, text_col)

    cmd = [
        *module_or_script_cmd("train_dann"),
        "--standardized-csv", str(STANDARDIZED_CSV),
        "--source-dataset", SOURCE_DATASET,
        "--target-dataset", target_dataset,
        "--source-image-roots", str(PADUFES_IMAGE_DIR),
        "--target-image-roots", *map(str, target_roots),
        "--checkpoint", str(ckpt),
        "--output-dir", str(DANN_OUT),
        "--model-family", model_family,
        "--text-col", text_col,
        "--batch-size", str(batch_size),
        "--epochs", str(epochs),
        "--energy-temperature", str(ENERGY_TEMPERATURE),
    ]
    run_command(cmd)

def validate_standardized_dataset_names():
    df = pd.read_csv(STANDARDIZED_CSV)

    available = set(df["dataset"].dropna().unique())
    required = {SOURCE_DATASET, *TARGETS.keys()}
    missing = required - available

    if missing:
        raise ValueError(
            "Dataset name mismatch in standardized CSV.\n"
            f"Missing requested names: {sorted(missing)}\n"
            f"Available names: {sorted(available)}\n\n"
            "Use the exact names from the standardized CSV."
        )

    print("Standardized dataset names OK:")
    print(sorted(available))

def run_full_concat_pipeline(
    model_family: str,
    n_trials: int = 10,
    trial_epochs: int = 3,
    closed_set_epochs: int = 50,
    dann_epochs: int = 20,
):
    assert model_family in {"mobilevit_concat", "resnet50_concat"}

    print("\n" + "#" * 120)
    print(f"STARTING PIPELINE: {model_family}")
    print("#" * 120)

    print(
        f"Settings: n_trials={n_trials}, "
        f"trial_epochs={trial_epochs}, "
        f"closed_set_epochs={closed_set_epochs}, "
        f"dann_epochs={dann_epochs}"
    )

    # 1. Hyperparameter tuning.
    tuning_dir = tune_closed_set(
        model_family=model_family,
        n_trials=n_trials,
        trial_epochs=trial_epochs,
    )
    best_params = load_best_hyperparameters(tuning_dir)
    print_tuning_trials(tuning_dir)

    # 2. Apply best parameters where scripts consume config.py defaults.
    apply_best_params_to_config(best_params)
    batch_size = batch_size_from_params(best_params)
    print(f"\nUsing batch size for {model_family}: {batch_size}")

    # 3. Closed-set training on all three text combinations.
    run_closed_set_training(
        model_family=model_family,
        batch_size=batch_size,
        epochs=closed_set_epochs,
    )

    # 4. Print closed-set results.
    print_closed_set_results(model_family)

    # 5. Direct transfer, DANN, and energy-based open-world evaluation.
    for text_col in TEXT_COLS:
        for target_dataset, target_roots in TARGETS.items():
            print("\n" + "-" * 120)
            print(f"{model_family} | {text_col} | target={target_dataset}")
            print("-" * 120)

            run_direct_generalization(
                model_family=model_family,
                text_col=text_col,
                target_dataset=target_dataset,
                target_roots=target_roots,
                batch_size=batch_size,
            )

            run_dann(
                model_family=model_family,
                text_col=text_col,
                target_dataset=target_dataset,
                target_roots=target_roots,
                batch_size=batch_size,
                epochs=dann_epochs,
            )

            run_open_world(
                model_family=model_family,
                text_col=text_col,
                target_dataset=target_dataset,
                target_roots=target_roots,
                batch_size=batch_size,
            )

    # 6. Print final summaries.
    print_all_results(model_family)

    print("\n" + "#" * 120)
    print(f"FINISHED PIPELINE: {model_family}")
    print("#" * 120)

In [28]:
validate_standardized_dataset_names()

Standardized dataset names OK:
['ISIC 2019', 'MCR-SL', 'PAD-UFES-20']


C:\Users\Nebula PC\AppData\Local\Temp\ipykernel_29264\2003150044.py:66: DtypeWarning: Columns (12,14,15,16,17,18,22,23,24,25,26,27,33,34,35,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(STANDARDIZED_CSV)


# Part A — MobileViT concatenation

This block performs:

- Hyperparameter tuning for `mobilevit_concat`
- Closed-set training using all three text combinations
- Direct generalization
- DANN
- Energy-based open-world evaluation
- Final open-world result printing

In [ ]:
# validate_project_files()
run_full_concat_pipeline(
    model_family="mobilevit_concat",
    n_trials=2,
    trial_epochs=1,
    closed_set_epochs=1,
    dann_epochs=1,
)


########################################################################################################################
STARTING PIPELINE: mobilevit_concat
########################################################################################################################
Settings: n_trials=2, trial_epochs=1, closed_set_epochs=1, dann_epochs=1

d:\anaconda3\envs\pytorch\python.exe -m src.tune_hyperparameters --study closed_set --padufes-csv D:\Deep Learning\metadata.csv --padufes-image-dir D:\Deep Learning\images --output-dir D:\Deep Learning\output_concat_demo\01_tuning\mobilevit_concat\closed_set --n-trials 2 --trial-epochs 1 --model-families mobilevit_concat --text-cols text_full text_core text_missing_explicit --batch-sizes 8 16 32 --fusion-dims 128 256 384 512 --use-sampler --use-weighted-loss

STDOUT:
Imported config module: D:\Deep Learning\skin_lesion_openworld\skin_lesion_openworld\config.py
Best value: 0.28178899751933456
Best params:
  lr: 8.46800857524833e-06
  weight

,number,state,value,lr,weight_decay,fusion_dim,balance_beta,model_family,text_col,batch_size
0,0,COMPLETE,0.281789,0.000008,0.000711,128,0.905750,mobilevit_concat,text_full,16
1,1,COMPLETE,0.140331,0.000003,0.000004,384,0.928832,mobilevit_concat,text_full,32


Updated config.py with selected tuning values.

Using batch size for mobilevit_concat: 16

d:\anaconda3\envs\pytorch\python.exe -m src.train_closed_set --padufes-csv D:\Deep Learning\metadata.csv --padufes-image-dir D:\Deep Learning\images --output-dir D:\Deep Learning\output_concat_demo\02_closed_set --model-family mobilevit_concat --text-col all --batch-size 16 --epochs 1

STDOUT:

Closed-set experiment: model_family=mobilevit_concat, text_col=text_full
{'epoch': 1, 'train_loss': 1.3423703226490298, 'lr': 0.0001, 'val_accuracy': 0.6723404255319149, 'val_macro_precision': 0.3169625246548323, 'val_macro_recall': 0.4542857142857143, 'val_macro_f1': 0.3729699362283633, 'val_weighted_precision': 0.49193419782617814, 'val_weighted_recall': 0.6723404255319149, 'val_weighted_f1': 0.5677263182379128, 'val_loss': 0.9709708054860433, 'val_macro_auc_ovr': nan, 'val_weighted_auc_ovr': 0.8584810929697971, 'val_macro_auc': nan, 'val_weighted_auc': 0.8584810929697971, 'val_auc': nan, 'val_precision_

,model_family,text_col,output_dir,best_checkpoint,root_best_checkpoint,final_checkpoint,best_val_f1_macro,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_precision,test_weighted_recall,test_weighted_f1,test_loss,test_macro_auc_ovr,test_weighted_auc_ovr,test_macro_auc,test_weighted_auc,test_auc,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,source_file
0,mobilevit_concat,text_full,D:\Deep Learning\output_concat_demo\02_closed_...,D:\Deep Learning\output_concat_demo\02_closed_...,D:\Deep Learning\output_concat_demo\02_closed_...,D:\Deep Learning\output_concat_demo\02_closed_...,0.37297,0.682203,0.350401,0.490714,0.407304,0.499086,0.682203,0.57526,0.929167,NaN,0.871782,NaN,0.871782,NaN,0.350401,0.490714,0.407304,0.499086,0.682203,0.57526,D:\Deep Learning\output_concat_demo\02_closed_...



------------------------------------------------------------------------------------------------------------------------
mobilevit_concat | text_full | target=ISIC 2019
------------------------------------------------------------------------------------------------------------------------

d:\anaconda3\envs\pytorch\python.exe -m src.evaluate_closed_set_transfer --standardized-csv D:\Deep Learning\output_concat_demo\standardized\all_preprocessed_splits_standardized_text.csv --target-dataset ISIC 2019 --target-image-roots D:\Deep Learning\ISIC_2019_Training_Input D:\Deep Learning\ISIC_2019_Test_Input --checkpoint D:\Deep Learning\output_concat_demo\02_closed_set\mobilevit_concat\text_full\best.pt --output-dir D:\Deep Learning\output_concat_demo\03_direct_transfer --model-family mobilevit_concat --text-col text_full --batch-size 16

STDOUT:
Saved direct closed-set transfer outputs to D:\Deep Learning\output_concat_demo\03_direct_transfer\mobilevit_concat\text_full\ISIC_2019\direct_closed

In [ ]:
from pathlib import Path
import ast
import re

SRC = Path(r"D:/Deep Learning/skin_lesion_openworld/skin_lesion_openworld/src")

def get_config_fields():
    tree = ast.parse((SRC / "config.py").read_text(encoding="utf-8"))
    fields = []
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == "ExperimentConfig":
            for item in node.body:
                if isinstance(item, ast.AnnAssign) and isinstance(item.target, ast.Name):
                    fields.append(item.target.id)
    return fields

config_fields = set(get_config_fields())

required_config_fields = {
    "num_workers",
    "image_model_name",
    "resnet_model_name",
    "text_model_name",
    "max_text_len",
    "patience",
    "lr",
    "weight_decay",
    "scheduler_factor",
    "scheduler_patience",
    "fusion_dim",
    "num_heads",
    "balance_beta",
    "use_soft_weighted_sampler",
    "use_weighted_loss",
    "domain_loss_weight",
    "dann_lambda_max",
    "energy_temperature",
    "freeze_backbones",
}

print("Missing config fields:")
print(sorted(required_config_fields - config_fields))

for filename in [
    "models.py",
    "train_closed_set.py",
    "train_dann.py",
    "evaluate_openworld.py",
    "evaluate_closed_set_transfer.py",
    "tune_hyperparameters.py",
]:
    text = (SRC / filename).read_text(encoding="utf-8")
    print("\n", filename)
    print("  mobilevit_concat:", "mobilevit_concat" in text)
    print("  resnet50_concat:", "resnet50_concat" in text)
    print("  freeze_backbones refs:", text.count("freeze_backbones"))
    print("  dropout refs:", text.count("dropout"))

# Specific energy-threshold issue in evaluate_openworld.py
openworld_text = (SRC / "evaluate_openworld.py").read_text(encoding="utf-8")
print("\nEnergy threshold clamp problem:")
print("  clamps to [0, 1]:", "candidates >= 0.0" in openworld_text and "candidates <= 1.0" in openworld_text)

# Part B — ResNet50 concatenation

This block performs the same complete demonstration using `resnet50_concat`.

In [ ]:
# validate_project_files()
run_full_concat_pipeline("resnet50_concat")

## 12. Final comparison: MobileViT concat vs ResNet50 concat

In [ ]:
def compare_open_world_models():
    rows = []

    for model_family in ["mobilevit_concat", "resnet50_concat"]:
        root = OPENWORLD_OUT / model_family
        json_df = collect_json_metrics(root, ["*metrics*.json", "openworld_metrics.json"])

        if json_df.empty:
            continue

        json_df["model_family"] = model_family
        rows.append(json_df)

    if not rows:
        print("No open-world metrics found for comparison yet.")
        return pd.DataFrame()

    df = pd.concat(rows, ignore_index=True, sort=False)

    metric_cols = [
        c for c in df.columns
        if any(key in c for key in [
            "open_macro_f1",
            "open_weighted_f1",
            "unknown_auroc",
            "unknown_recall",
            "unknown_f1",
            "oscr",
            "threshold",
            "scoring_method",
        ])
    ]

    display_cols = ["model_family", "source_file", *metric_cols]
    display(df[display_cols].head(100))

    return df

comparison_df = compare_open_world_models()

## 13. Presentation notes

For a quick live demonstration, reduce runtime first:

```python
N_TRIALS = 2
TRIAL_EPOCHS = 1
CLOSED_SET_EPOCHS = 1
DANN_EPOCHS = 1
```

Then set:

```python
RUN_COMMANDS = True
```

For final experiments, increase trials and epochs again.